In [3]:
import os

from debugpy.common import timestamp
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langsmith import client
from langchain_deepseek import ChatDeepSeek
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AIMessage
from pydantic import Field, BaseModel
from dataclasses import field, dataclass
from dotenv import load_dotenv
from datetime import datetime
import json
from langchain.chat_models import init_chat_model
from typing import Literal
from langchain.embeddings import Embeddings

In [2]:
_ = load_dotenv()
llm = init_chat_model("deepseek-chat")

In [4]:
class ResearchSchema(BaseModel):
    answer: str = Field(description = "The answer to the question")
    confidence: Literal["high", "medium", "low"] = Field(description = "How confident are you in the answer")
    sources: list[str] = Field(description= "A list of sources the answer came from")
    key_quotes: list[str] = Field(description = "Relevant quotes from the source", default = [])
    follow_up: list[str] = Field(description = "Suggested follow up questions", default = [])



In [8]:
from google.genai import Client as genaiClient
from google.genai import types

class GeminiEmbeddings(Embeddings):
    def __init__(self, model_name: str = "gemini-embedding-2", dimensions: int = 3072):
        self.model_name = model_name
        self.dimensions = dimensions
        self.client = genaiClient()

    def _embed(self, text: str) -> list[float]:
        result = self.client.models.embed_content(
            model= self.model_name,
            contents = text,
            config=types.EmbedContentConfig(output_dimensionality=self.dimensions),
        )
        return result.embeddings[0].values

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

embeddings = GeminiEmbeddings()

In [15]:
from typing import Any
@dataclass
class ResearchAssistant:
    persist_directory: str = "./research_db"
    chunk_size: int = 1000
    chunk_overlap: int = 100

    _embeddings: Any = field(default = None, init=False, repr=False)
    _vectorstore: Any = field(default =  None, init = False, repr=False)
    def __post_init__(self):
        self.splitter = RecursiveCharacterTextSplitter(chunk_size = self.chunk_size, chunk_overlap=self.chunk_overlap)

    def add_documents(self, documents: list[Document], source_name: str | None = None) -> int:
        if source_name:
            for doc in documents:
                doc.metadata["source"] = source_name

        chunks = self.splitter.split_documents(documents)

        # Timestamp each chunk
        for chunk in chunks:
            chunk.metadata["indexed_at"] = datetime.now().isoformat()
        self.load_vectorstore.add_documents(chunks)

        print(f"Added {len(chunks)} chunks from {len(documents)} documents")
        return len(chunks)

    def add_text(self, text: str, source: str, metadata: dict | None = None) -> int:
        doc = Document(page_content=text, metadata={"source": source, **(metadata or {})})
        return self.add_documents([doc])

    def add_texts(self, texts: list[str], source: str) -> int:
        docs = [Document(page_content = t, metadata={"source": source}) for t in texts]
        return self.add_documents(docs)

    def get_document_count(self) -> int:
        return self.load_vectorstore._collection.count()

    def list_sources(self) -> list[str]:
        results = self.load_vectorstore._collection.get()
        metadatas = results.get("metadatas") or []
        return sorted({m["source"] for m in metadatas if m and "source" in m})

    @property
    def embeddings(self):
        if self._embeddings is None:
            self._embeddings = GeminiEmbeddings()
        return self._embeddings

    @property
    def load_vectorstore(self):
        if self._vectorstore is None:
            self._vectorstore = Chroma(persist_directory=self.persist_directory, embedding_function = self.embeddings, collection_name="research_docs")
        return self._vectorstore